# Scaffold-based 5-Fold Cross-Validation

- SOURCE: use everything (fixed)
- TARGET: 5-Fold Scaffold Split (train/test)

For each fold:
- fold_0: source + train_fold_0, test_fold_0
- fold_1: source + train_fold_1, test_fold_1
- ...
- fold_4: source + train_fold_4, test_fold_4

In [10]:
import pandas as pd
import os
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold

RDLogger.DisableLog("rdApp.*")

# SOURCE: smart deduplicated (only duplicates removed, leakage kept)
source_dir = r'../data/in_vitro_data_deduplicated'
target_dir = r'../data/pksmarts_data/final'
output_dir = r'../data/cv_datasets'
os.makedirs(output_dir, exist_ok=True)

# Property mapping: (property_name, [source_files], target_type)
PROPERTIES = [
    ('clearance', 
     ['chembl_CL_raw_smart_deduplicated.csv', 'biogen_CL_raw_smart_deduplicated.csv']),
    ('half_life',
     ['tdc_half_life_smart_deduplicated.csv']),
    ('fu',
     ['chembl_Fu_raw_smart_deduplicated.csv']),
]

N_FOLDS = 5

smiles_col_candidates = ['smiles', 'SMILES', 'Smiles', 'Drug']

def get_smiles_column(df):
    for col in smiles_col_candidates:
        if col in df.columns:
            return col
    return None

def get_scaffold(smi):
    """Get Bemis-Murcko scaffold"""
    try:
        m = Chem.MolFromSmiles(str(smi))
        if m is None:
            return None
        return MurckoScaffold.MurckoScaffoldSmiles(mol=m)
    except:
        return None

def scaffold_k_fold_split(df, k=5, seed=0):
    """Split data into k folds by scaffold
    
    Returns:
        list of (train_df, test_df) tuples
    """
    np.random.seed(seed)
    
    # Get SMILES column
    smiles_col = get_smiles_column(df)
    if smiles_col is None:
        raise ValueError("No SMILES column found")
    
    # Calculate scaffolds
    df = df.copy()
    df['scaffold'] = df[smiles_col].map(get_scaffold)
    df = df.dropna(subset=['scaffold'])
    
    # Group by scaffold
    scaffolds = df['scaffold'].unique()
    scaffold_indices = {s: df[df['scaffold'] == s].index.tolist() for s in scaffolds}
    
    # Assign scaffolds to folds (round-robin, largest first)
    scaffold_sizes = [(s, len(idx)) for s, idx in scaffold_indices.items()]
    scaffold_sizes.sort(key=lambda x: -x[1])  # largest first
    
    fold_indices = [[] for _ in range(k)]
    for scaffold, _ in scaffold_sizes:
        # Assign to fold with smallest current size
        smallest_fold = min(range(k), key=lambda i: len(fold_indices[i]))
        fold_indices[smallest_fold].extend(scaffold_indices[scaffold])
    
    # Create train/test splits
    splits = []
    for test_fold_idx in range(k):
        test_indices = fold_indices[test_fold_idx]
        train_indices = []
        for i in range(k):
            if i != test_fold_idx:
                train_indices.extend(fold_indices[i])
        
        train_df = df.iloc[train_indices].drop('scaffold', axis=1).reset_index(drop=True)
        test_df = df.iloc[test_indices].drop('scaffold', axis=1).reset_index(drop=True)
        splits.append((train_df, test_df))
    
    return splits

print("Scaffold-based K-Fold CV Split Processing (NO LEAKAGE REMOVAL)...\n")
print("Keep compounds shared between SOURCE and TARGET (for domain adaptation)\\n")

Scaffold-based K-Fold CV Split Processing (NO LEAKAGE REMOVAL)...

Keep compounds shared between SOURCE and TARGET (for domain adaptation)\n


In [11]:
summary_data = []

for prop_name, src_files in PROPERTIES:
    print(f"\n{'='*120}")
    print(f"Property: {prop_name.upper()}")
    print(f"{'='*120}")
    
    # Load & combine SOURCE files
    source_dfs = []
    for src_file in src_files:
        src_path = os.path.join(source_dir, src_file)
        if not os.path.exists(src_path):
            print(f"  ⚠️ {src_file} not found")
            continue
        
        src_df = pd.read_csv(src_path)
        source_dfs.append(src_df)
        print(f"  ✓ {src_file}: {len(src_df)} rows")
    
    if not source_dfs:
        print(f"✗ No source files found for {prop_name}")
        continue
    
    source_df = pd.concat(source_dfs, ignore_index=True)
    print(f"Combined Source: {len(source_df)} rows")
    
    # Combine TARGET splits (train/val/test) into one
    target_files = [
        f"{prop_name}_target_train.csv",
        f"{prop_name}_target_val.csv",
        f"{prop_name}_target_test.csv"
    ]
    
    target_dfs = []
    for tgt_file in target_files:
        tgt_path = os.path.join(target_dir, tgt_file)
        if not os.path.exists(tgt_path):
            print(f"  ⚠️ {tgt_file} not found")
            continue
        
        tgt_df = pd.read_csv(tgt_path)
        target_dfs.append(tgt_df)
        print(f"  ✓ {tgt_file}: {len(tgt_df)} rows")
    
    if not target_dfs:
        print(f"✗ No target files found for {prop_name}")
        continue
    
    # Combine all target
    combined_target = pd.concat(target_dfs, ignore_index=True)
    print(f"Combined Target: {len(combined_target)} rows")
    
    # K-Fold Split
    print(f"\nScaffold-based {N_FOLDS}-Fold Split:")
    splits = scaffold_k_fold_split(combined_target, k=N_FOLDS, seed=42)
    
    # Save each fold
    fold_summary = []
    for fold_idx, (train_df, test_df) in enumerate(splits):
        # Create fold directory
        fold_dir = os.path.join(output_dir, f"fold_{fold_idx}")
        os.makedirs(fold_dir, exist_ok=True)
        
        # Save SOURCE (same for all folds, no leakage removal)
        src_save_path = os.path.join(fold_dir, f"{prop_name}_source.csv")
        source_df.to_csv(src_save_path, index=False)
        
        # Save TARGET train/test
        train_save_path = os.path.join(fold_dir, f"{prop_name}_target_train.csv")
        test_save_path = os.path.join(fold_dir, f"{prop_name}_target_test.csv")
        train_df.to_csv(train_save_path, index=False)
        test_df.to_csv(test_save_path, index=False)
        
        print(f"  Fold {fold_idx}: train {len(train_df):5d} | test {len(test_df):4d}")
        
        fold_summary.append({
            'Fold': fold_idx,
            'Property': prop_name,
            'Source': len(source_df),
            'Target_Train': len(train_df),
            'Target_Test': len(test_df),
            'Target_Total': len(train_df) + len(test_df)
        })
    
    summary_data.extend(fold_summary)


Property: CLEARANCE
  ✓ chembl_CL_raw_smart_deduplicated.csv: 79821 rows
  ✓ biogen_CL_raw_smart_deduplicated.csv: 3521 rows
Combined Source: 83342 rows
  ⚠️ clearance_target_train.csv not found
  ⚠️ clearance_target_val.csv not found
  ⚠️ clearance_target_test.csv not found
✗ No target files found for clearance

Property: HALF_LIFE
  ✓ tdc_half_life_smart_deduplicated.csv: 665 rows
Combined Source: 665 rows
  ⚠️ half_life_target_train.csv not found
  ⚠️ half_life_target_val.csv not found
  ⚠️ half_life_target_test.csv not found
✗ No target files found for half_life

Property: FU
  ✓ chembl_Fu_raw_smart_deduplicated.csv: 14011 rows
Combined Source: 14011 rows
  ⚠️ fu_target_train.csv not found
  ⚠️ fu_target_val.csv not found
  ⚠️ fu_target_test.csv not found
✗ No target files found for fu


## Final summary

In [12]:
summary_df = pd.DataFrame(summary_data)

print("\n" + "="*120)
print("5-FOLD SCAFFOLD CV SUMMARY")
print("="*120)
print(summary_df.to_string(index=False))

print(f"\n{'='*120}")
print(f"Output Directory: {output_dir}")
print(f"{'='*120}")
print(f"""
Directory Structure:
  cv_datasets/
  ├─ fold_0/
  │  ├─ clearance_source.csv
  │  ├─ clearance_target_train.csv
  │  ├─ clearance_target_test.csv
  │  ├─ half_life_source.csv
  │  ├─ half_life_target_train.csv
  │  ├─ half_life_target_test.csv
  │  ├─ fu_source.csv
  │  ├─ fu_target_train.csv
  │  └─ fu_target_test.csv
  ├─ fold_1/
  ├─ fold_2/
  ├─ fold_3/
  └─ fold_4/

Training flow:
  1. Repeat for each fold:
     - load source (fixed, use everything)
     - load target_train (for unsupervised alignment)
     - train the model
     - evaluate on target_test
  
  2. Final performance = mean/std of the 5 folds
""")

# Save
summary_df.to_csv(os.path.join(output_dir, 'cv_summary.csv'), index=False)


5-FOLD SCAFFOLD CV SUMMARY
Empty DataFrame
Columns: []
Index: []

Output Directory: ..\data\cv_datasets

Directory Structure:
  cv_datasets/
  ├─ fold_0/
  │  ├─ clearance_source.csv
  │  ├─ clearance_target_train.csv
  │  ├─ clearance_target_test.csv
  │  ├─ half_life_source.csv
  │  ├─ half_life_target_train.csv
  │  ├─ half_life_target_test.csv
  │  ├─ fu_source.csv
  │  ├─ fu_target_train.csv
  │  └─ fu_target_test.csv
  ├─ fold_1/
  ├─ fold_2/
  ├─ fold_3/
  └─ fold_4/

Training flow:
  1. Repeat for each fold:
     - load source (fixed, use everything)
     - load target_train (for unsupervised alignment)
     - train the model
     - evaluate on target_test
  
  2. Final performance = mean/std of the 5 folds

